# Conditional Generative Pretrained Transformer (GPT)

This notebook demonstrates generation of molecules using a GPT model conditioned on $logP$ and SAS values

## Setting up the notebook and importing necessary libraries

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import QED, Descriptors
from rdkit.Chem import RDConfig
from rdkit import RDLogger
import os
import sys
sys.path.append(os.path.join(RDConfig.RDContribDir, 'SA_Score'))
import sascorer
import re
import math
import random
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

lg = RDLogger.logger()
lg.setLevel(RDLogger.ERROR)
RDLogger.DisableLog('rdApp.*')

SMILES_TOKEN_PATTERN = re.compile(r"(\[[^\]]+]|Br?|Cl?|N|O|S|P|F|I|b|c|n|o|s|p|\(|\)|\.|=|#|-|\+|\\|\/|:|~|@|\?|>>?|\*|\$|\%[0-9]{2}|[0-9])")

In [ ]:
class ZincDataset(Dataset):
    def __init__(self, data, block_size, stoi=None, itos=None):
        self.data = data
        self.max_len = block_size
        
        if stoi is not None and itos is not None:
            self.stoi = stoi
            self.itos = itos
            self.vocab_size = len(self.stoi)
        else:
            all_tokens = set()
            for smiles in tqdm(data['smiles'], desc='Building vocab'):
                tokens = SMILES_TOKEN_PATTERN.findall(smiles)
                all_tokens.update(tokens)
            all_tokens.add('<')
            self.vocab = sorted(list(all_tokens))
            self.stoi = {ch:i for i,ch in enumerate(self.vocab)}
            self.itos = {i:ch for i,ch in enumerate(self.vocab)}
            self.vocab_size = len(self.vocab)
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        smiles = row['smiles'].strip()
        props = torch.tensor([row['logP'], row['SAS']], dtype=torch.float)
        
        tokens = SMILES_TOKEN_PATTERN.findall(smiles)
        tokens += ['<'] * (self.max_len - len(tokens))
        tokens = tokens[:self.max_len]
        
        dix = [self.stoi[s] for s in tokens]
        
        return torch.tensor(dix[:-1], dtype=torch.long), torch.tensor(dix[1:], dtype=torch.long), props

def load_data(file_path):
    data = pd.read_csv(file_path)
    lens = [len(SMILES_TOKEN_PATTERN.findall(s)) for s in tqdm(data['smiles'], desc='Calculating lengths')]
    return data, max(lens)

In [ ]:
try:
    data, max_len = load_data('../../datasets/zinc250k.csv')
except FileNotFoundError:
    data, max_len = load_data('zinc250k.csv')

print('Building global vocabulary...') 
dummy_ds = ZincDataset(data, max_len) # Build vocab from full data
stoi, itos = dummy_ds.stoi, dummy_ds.itos

train_data, test_data = train_test_split(data, test_size=0.1, random_state=42)

train_dataset = ZincDataset(train_data.reset_index(drop=True), max_len, stoi=stoi, itos=itos)
test_dataset  = ZincDataset(test_data.reset_index(drop=True), max_len, stoi=stoi, itos=itos)

print(f'Train: {len(train_dataset)}, Test: {len(test_dataset)}')
print(f'Vocab size: {len(stoi)}')

## Load dataset, tokenize SMILES and make PyTorch Dataloaders

## Defining Elements of the model architecture

### Self-Attention and Transformer Blocks

*Causal Self Attention* which is a form of Masked Self Attention is used where only the tokens from the past until the present are used to predict the next token. Each block is a unit of the Conditional GPT model which is repeated several times in the architecture.

In [4]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.key = nn.Linear(config.n_embd, config.n_embd)
        self.query = nn.Linear(config.n_embd, config.n_embd)
        self.value = nn.Linear(config.n_embd, config.n_embd)
        self.attn_drop = nn.Dropout(config.attn_pdrop)
        self.resid_drop = nn.Dropout(config.resid_pdrop)
        self.proj = nn.Linear(config.n_embd, config.n_embd)
        self.n_head = config.n_head
        
        # Causal mask with space for property embeddings
        self.register_buffer("mask", torch.tril(torch.ones(config.block_size + 1, config.block_size + 1))
                             .view(1, 1, config.block_size + 1, config.block_size + 1))
    
    def forward(self, x):
        B, T, C = x.size()
        k = self.key(x).view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = self.query(x).view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = self.value(x).view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.mask[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_drop(self.proj(y))
        return y

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.ln2 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.mlp = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(),
            nn.Linear(4 * config.n_embd, config.n_embd),
            nn.Dropout(config.resid_pdrop),
        )
    
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

### Defining the GPT model

In [5]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.tok_emb = nn.Embedding(config.vocab_size, config.n_embd)
        self.pos_emb = nn.Parameter(torch.zeros(1, config.block_size, config.n_embd))
        self.drop = nn.Dropout(config.embd_pdrop)
        self.blocks = nn.Sequential(*[Block(config) for _ in range(config.n_layer)])
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.prop_nn = nn.Linear(2, config.n_embd)  # For logP and SAS
        
        self.apply(self._init_weights)
        print("number of parameters:", sum(p.numel() for p in self.parameters()))
    
    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            module.weight.data.normal_(mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, nn.LayerNorm):
            module.bias.data.zero_()
            module.weight.data.fill_(1.0)
    
    def get_block_size(self):
        return self.config.block_size
    
    def forward(self, idx, targets=None, prop=None):
        b, t = idx.size()
        assert t <= self.config.block_size, "Cannot forward, model block size is exhausted."
        
        token_embeddings = self.tok_emb(idx)
        position_embeddings = self.pos_emb[:, :t, :]
        x = self.drop(token_embeddings + position_embeddings)
        
        if prop is not None:
            p = self.prop_nn(prop.unsqueeze(1))
            x = torch.cat([p, x], 1)
        
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)
        
        if prop is not None:
            logits = logits[:, 1:, :]
        
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.view(-1), ignore_index=self.config.ignore_index)

        
        return logits, loss

## Training the model

In [6]:
def train(model, train_dataset, batch_size=128, epochs=10, lr=3e-4, save_path='conditional_gpt.pt'):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    
    train_losses = []
    
    for epoch in range(epochs):
        # --- Train ---
        model.train()
        epoch_train_losses = []
        pbar = tqdm(enumerate(train_loader), total=len(train_loader))
        for it, (x, y, props) in pbar:
            x, y, props = x.to(device), y.to(device), props.to(device)
            logits, loss = model(x, y, props)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_train_losses.append(loss.item())
            pbar.set_description(f"epoch {epoch+1} iter {it}: loss {loss.item():.5f}")
        
        avg_train = np.mean(epoch_train_losses)
        train_losses.append(avg_train)
        print(f"Epoch {epoch+1}: train_loss={avg_train:.4f}")
    
    # --- Save checkpoint ---
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'config': model.config,
        'vocab': {
            'stoi': train_dataset.stoi,
            'itos': train_dataset.itos,
            'vocab_size': train_dataset.vocab_size,
        },
        'max_len': train_dataset.max_len,
        'train_losses': train_losses,
    }
    torch.save(checkpoint, save_path)
    print(f"\nModel saved to {save_path}")
    
    return train_losses

#### Defining the Config file for the model

In [7]:
class GPTConfig:
    def __init__(self, vocab_size, block_size, ignore_index=-1, **kwargs):
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.ignore_index = ignore_index
        self.embd_pdrop = 0.1
        self.resid_pdrop = 0.1
        self.attn_pdrop = 0.1
        self.n_layer = 8
        self.n_head = 8
        self.n_embd = 256
        for k,v in kwargs.items():
            setattr(self, k, v)

### Training call: Skip the cell below if loading a pt file.

In [ ]:
config = GPTConfig(train_dataset.vocab_size, max_len, num_props=2, ignore_index=train_dataset.stoi['<'])
model = GPT(config).to(device)

train_losses = train(model, train_dataset, batch_size=128, epochs=10)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(train_losses)+1), train_losses, 'o-', label='Train Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training Loss')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

number of parameters: 6369024


epoch 1 iter 1559: loss 0.70755: 100%|██████████| 1560/1560 [03:34<00:00,  7.29it/s]


Epoch 1: train_loss=0.9817  val_loss=7.1289


epoch 2 iter 1559: loss 0.67944: 100%|██████████| 1560/1560 [04:08<00:00,  6.29it/s]


Epoch 2: train_loss=0.7233  val_loss=6.8327


...

epoch 4 iter 1559: loss 0.60944: 100%|██████████| 1560/1560 [05:28<00:00,  4.75it/s]


Epoch 4: train_loss=0.6292  val_loss=7.7655


epoch 5 iter 1559: loss 0.61318: 100%|██████████| 1560/1560 [05:52<00:00,  4.42it/s]


Epoch 5: train_loss=0.6052  val_loss=8.0270


epoch 6 iter 1559: loss 0.59099: 100%|██████████| 1560/1560 [06:01<00:00,  4.32it/s]


### Loading Model

In [ ]:
CHECKPOINT_PATH = 'conditional_gpt.pt'

if os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    model = GPT(checkpoint['config']).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    train_losses = checkpoint.get('train_losses', [])
    print(f"Loaded model from {CHECKPOINT_PATH}")
else:
    print(f"Checkpoint not found at {CHECKPOINT_PATH}. Please train the model first.")

## Generating molecules

In [ ]:
def sample(model, x, steps, temperature=1.0, sample=True, top_k=None, prop=None):
    block_size = model.get_block_size()
    model.eval()
    
    with torch.no_grad():
        for k in range(steps):
            x_cond = x if x.size(1) <= block_size else x[:, -block_size:]
            logits, _ = model(x_cond, prop=prop)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, ix = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            if sample:
                ix = torch.multinomial(probs, num_samples=1)
            else:
                _, ix = torch.topk(probs, k=1, dim=-1)
            x = torch.cat((x, ix), dim=1)
    
    return x

In [ ]:
# defining property ranges
logp_range = (data['logP'].min(), data['logP'].max())
sas_range = (data['SAS'].min(), data['SAS'].max())

#number of target conditions and number of generations per condition
num_conditions = 100
num_samples_per_condition = 10

generated_smiles = []
generated_targets = []

for _ in tqdm(range(num_conditions), desc="Generating conditions"):
    target_logp = np.random.uniform(*logp_range)
    target_sas = np.random.uniform(*sas_range)
    
    for _ in range(num_samples_per_condition):
        context = "C"
        x = torch.tensor([train_dataset.stoi[s] for s in SMILES_TOKEN_PATTERN.findall(context)], dtype=torch.long)[None,...].to(device)
        props = torch.tensor([target_logp, target_sas], dtype=torch.float)[None,...].to(device)
        
        y = sample(model, x, steps=100, temperature=1.0, prop=props)
        completion = ''.join([train_dataset.itos[int(i)] for i in y[0]]).replace('<', '')
        
        generated_smiles.append(completion)
        generated_targets.append((target_logp, target_sas))

print(f"\nGenerated {len(generated_smiles)} molecules.")

## Evaluating Model Performance

In [ ]:
tr_data = set(data['smiles'])

tol_logp = 0.4
tol_sas = 0.5

results = []

# tracking accuracy
both_in_limit = 0
logp_in_limit = 0
sas_in_limit = 0
valid_count = 0

logp_errors = []
sas_errors = []

# tracking novelty and uniqueness
unique_smiles = set()
unique_novel_smiles = set()

generated_logps = []
generated_sass = []

for smiles, (target_logp, target_sas) in tqdm(zip(generated_smiles, generated_targets), 
                                              total=len(generated_smiles), desc="Evaluating"):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        continue

    try:
        logp = Descriptors.MolLogP(mol)
        qed = QED.qed(mol)
        sa = sascorer.calculateScore(mol)

        logp_error = abs(logp - target_logp)
        sas_error = abs(sa - target_sas)

        logp_ok = logp_error <= tol_logp
        sas_ok = sas_error <= tol_sas
        both_ok = logp_ok and sas_ok

        # Update counters
        if logp_ok:
            logp_in_limit += 1
        if sas_ok:
            sas_in_limit += 1
        if both_ok:
            both_in_limit += 1

        valid_count += 1
        logp_errors.append(logp_error)
        sas_errors.append(sas_error)

        generated_logps.append(logp)
        generated_sass.append(sa)

        # Track uniqueness
        unique_smiles.add(smiles)

        # Track novelty correctly
        if smiles not in tr_data:
            unique_novel_smiles.add(smiles)

        # Store result
        results.append({
            "SMILES": smiles,
            "Target_logP": target_logp,
            "Target_SAS": target_sas,
            "logP": logp,
            "SAS": sa,
            "QED": qed,
            "logP_Error": logp_error,
            "SAS_Error": sas_error,
            "logP_in_limit": logp_ok,
            "SAS_in_limit": sas_ok,
            "Both_in_limit": both_ok
        })

    except Exception as e:
        continue

# calculating metrics
total_generated = len(generated_smiles)
validity = valid_count / total_generated * 100 if total_generated else 0

accuracy_both = both_in_limit / valid_count * 100 if valid_count else 0
accuracy_logp = logp_in_limit / valid_count * 100 if valid_count else 0
accuracy_sas = sas_in_limit / valid_count * 100 if valid_count else 0

mae_logp = np.mean(logp_errors) if logp_errors else None
mae_sas = np.mean(sas_errors) if sas_errors else None

# Novelty
novelty = len(unique_novel_smiles) / len(unique_smiles) * 100 if unique_smiles else 0

# Uniqueness
uniqueness = len(unique_smiles) / valid_count * 100 if valid_count else 0

# metrics reporting
print(f"Validity: {validity:.2f}% ({valid_count} / {total_generated})")
print(f"Accuracy (Both in limit): {accuracy_both:.2f}% ({both_in_limit} / {valid_count})")
print(f"Accuracy (logP in ±{tol_logp}): {accuracy_logp:.2f}% ({logp_in_limit} / {valid_count})")
print(f"Accuracy (SAS in ±{tol_sas}): {accuracy_sas:.2f}% ({sas_in_limit} / {valid_count})")
print(f"MAE logP: {mae_logp:.4f}")
print(f"MAE SAS: {mae_sas:.4f}")
print(f"Novelty: {novelty:.2f}% ({len(unique_novel_smiles)} / {len(unique_smiles)})")
print(f"Uniqueness: {uniqueness:.2f}% ({len(unique_smiles)} / {valid_count})")

# save the generated molecules
if results:
    df = pd.DataFrame(results)
    df.to_csv("conditional_generation_results.csv", index=False)

## Result Visualizations

In [ ]:
# --- Scatter Plots: Target vs Actual Properties ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

df = pd.DataFrame(results)

# logP scatter
ax = axes[0]
ax.scatter(df['Target_logP'], df['logP'], alpha=0.4, s=15, c='#4C72B0')
lims = [min(df['Target_logP'].min(), df['logP'].min()) - 0.5,
        max(df['Target_logP'].max(), df['logP'].max()) + 0.5]
ax.plot(lims, lims, '--', color='red', linewidth=1, label='Ideal (y = x)')
ax.set_xlabel('Target logP')
ax.set_ylabel('Actual logP')
ax.set_title(f'logP: Target vs Actual  (MAE={mae_logp:.3f})')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

# SAS scatter
ax = axes[1]
ax.scatter(df['Target_SAS'], df['SAS'], alpha=0.4, s=15, c='#DD8452')
lims = [min(df['Target_SAS'].min(), df['SAS'].min()) - 0.5,
        max(df['Target_SAS'].max(), df['SAS'].max()) + 0.5]
ax.plot(lims, lims, '--', color='red', linewidth=1, label='Ideal (y = x)')
ax.set_xlabel('Target SAS')
ax.set_ylabel('Actual SAS')
ax.set_title(f'SAS: Target vs Actual  (MAE={mae_sas:.3f})')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

In [ ]:
# --- Error Distribution Histograms ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(logp_errors, bins=30, color='#4C72B0', alpha=0.7, edgecolor='black')
axes[0].axvline(x=tol_logp, color='red', linestyle='--', label=f'Tolerance (±{tol_logp})')
axes[0].set_xlabel('|logP Error|')
axes[0].set_ylabel('Count')
axes[0].set_title('logP Absolute Error Distribution')
axes[0].legend()

axes[1].hist(sas_errors, bins=30, color='#DD8452', alpha=0.7, edgecolor='black')
axes[1].axvline(x=tol_sas, color='red', linestyle='--', label=f'Tolerance (±{tol_sas})')
axes[1].set_xlabel('|SAS Error|')
axes[1].set_ylabel('Count')
axes[1].set_title('SAS Absolute Error Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()